In [48]:
import pandas as pd
import os
from config import CENSUS_INPUT_DIR, CENSUS_OUTPUT_DIR

#DATA_ROOT = Path(r"C:\Users\johnh\Desktop\Datasets\_6_Demographics\US_Census")
folder_names = os.listdir(CENSUS_INPUT_DIR)
# OR USE THIS 
# folder_names = [p.name for p in INPUT_DIR.iterdir() if p.is_dir()]
print(*folder_names, sep='\n')


replacements = {
    ":": "",
    " years":"y",
    " year":"y",
    "different": "diff.",
    "Estimate!!": "",
    "Median age!!": "",
    "Median age --!!":"",
    "Total:!!": "",
    "Total!!": "",
    "United States": "US",
    "Average household size --!!":"",
    "Average household size!!":"",
    "Moved from": "",
    " (dollars)":"",
    "1, detached or attached": "1unit",
    "2 to 4" : "2-4units",
    "5 to 19" : "5-19units",
    "20 to 49" : "20-49units",
    "50 or more" : "50+units",
}
#   "" : "",

B08136_Commute_Time_By_Mode
B08203_Vehicles_Per_Worker
B08301_Commute_count_by_mode
B11001_Household_Type
B25001_Housing_Units
B25002_Occupancy_Status
B25004_Vacancy_Status
B25005_Vacant_Current_Residence_Elsewhere
B25008_Total_Population_By_Tenure
B25010_Average_Household_Size_By_Tenure
B25014_Occupants_Per_Room
B25017_Room_Count
B25018_Median_Room_Count
B25019_Aggregate_Room_Count
B25021_Median_Room_Count_By_Tenure
B25024_Units_In_Structure
B25031_Median_Rent_By_Bedroom_Count
B25032_Tenure_By_Units_In_Structure
B25037_Median_Building_Age_By_Tenure
B25039_Median_Duration_Lived_By_Tenure
B25044_Vehicles_Available_By_Tenure
B25052_Kitchen_Facilities
B25058_Median_Contract_Rent
B25064_Median_Gross_Rent
B25066_Aggregate_Gross_Rent_By_Units_In_Structure
B25069_Inclusion_of_Utilities_In_Rent
B25071_Median_Gross_Rent_Burden
B25075_Value
B25077_Median_Value
B25080_Aggregate_Value_By_Units_In_Structure
B25105_Median_Monthly_Housing_Costs
B25111_Median_Gross_Rent_By_Decade_Built
B25113_Median_G

In [64]:
#John Nabors Jan 23 2026

singleFolderOverride = True #allows manual processing of individual datasets
singleFolderName = "B25080_Aggregate_Value_By_Units_In_Structure"

for folder_name in folder_names: #loop all folders in /input
    if singleFolderOverride: folder_name = singleFolderName
    #print("parsing folder: ", folder_name)
    

    folder_path = CENSUS_INPUT_DIR / folder_name
    data_filenames = [file for file in os.listdir(folder_path) if file.endswith("Data.csv")]
    #print(*data_filenames, sep='\n')
    data_filepaths = [p for p in folder_path.glob("*-Data.csv")]
    #print(*data_filepaths, sep='\n')
    
    
    allYears = pd.DataFrame()
    for csv in data_filepaths: #loop all csvs in a folder.
        try   : df = pd.read_csv(csv, low_memory=False)
        except: print(f"Error reading csv: {csv}")
        #print(df.columns)
        
        year = str(csv).split('ACSDT5Y')[1][:4] #only works for specific filenaming scheme given by Census. TODO fix if i want to generalize script.
        #print("Data Year: ",year)
        
    

        # clean up column name formatting.
        df = df.drop(columns=["GEO_ID"]) #remove GEOID
        df.columns = df.iloc[0] #swap column headers with the first row column descriptors/human readable names
        #print(df.columns)
        df = df[1:].reset_index(drop=True) #not sure what this does
        if pd.isna(df.columns[-1]): df = df.drop(df.columns[-1], axis=1) #drop the na column artifact (WHY DOES IT APPEAR??)
        df = df.loc[:, ~df.columns.str.contains('Margin of Error', na=False)] #drop MOE columns 

        df.columns = df.columns.str.replace('|'.join(replacements.keys()), lambda m: replacements[m.group()], regex=True)
        df.columns = df.columns.str.replace('|'.join(replacements.keys()), lambda m: replacements[m.group()], regex=True)
        df.columns = df.columns.str.replace('|'.join(replacements.keys()), lambda m: replacements[m.group()], regex=True) # in case some are missed
        df.columns = df.columns + '_' + year

        df.rename(columns={f'Geographic Area Name_{year}': 'Geographic_Area_Name'}, inplace=True)
        
        #print(df.columns)
        

        if len(allYears) == 0:  allYears = df.copy()
        else: allYears = allYears.merge(df, on='Geographic_Area_Name', how='outer')
    
    allYears['zipCode'] = allYears['Geographic_Area_Name'].str[-5:]
    allYears = allYears[allYears['zipCode'].astype(int) > 1000] #removes puerto rico and territory zip codes
    allYears = allYears[['Geographic_Area_Name', 'zipCode'] + sorted([col for col in allYears.columns if col not in ['Geographic_Area_Name', 'zipCode'] ])]
    
    #print("AllYears: ", allYears.columns)
    ##output_name = "_6_Demographics/US_Census/Output/"+ folder_name + "joined.csv"
    output_name = f"Census/Output/{folder_name}_joined_v3.csv"
    print("outputting at: ",output_name)
    allYears.to_csv(output_name, index=False) #TODO make zip code index


    if singleFolderOverride: break
    


outputting at:  Census/Output/B25080_Aggregate_Value_By_Units_In_Structure_joined_v3.csv
